# Emoji provider classification
Predict the **provider** (apple, samsung, etc.), rather than emoji meaning.

Run cells in order. For the stronger model, use **Google Colab → Runtime → Change runtime type → GPU**. Upload this notebook, `baseline.py`, `train_labels.csv`, and the extracted `train/` and `test/` folders into the same working directory (or set DATA_DIR below to your Google Drive folder). For a ZIP, extract it first with `zipfile.ZipFile(path).extractall(destination)` using a trusted competition archive.

The CPU baseline is quick to understand; the pretrained ResNet18 learns visual style and should be evaluated against it. Accuracy on a held-out split is an estimate, not a promised Kaggle score. Run the required Colab submission individually for each teammate. Write your own report and explain the decisions and observed results.


In [ ]:
# Colab installation: Python 3.11–3.12 recommended for these PyTorch versions.
%pip install -q torch==2.6.0 torchvision==0.21.0 scikit-learn==1.9.1 pandas==2.3.3 numpy==2.2.6 pillow==11.3.0 matplotlib==3.10.7


In [ ]:
from pathlib import Path
import os, random, json, copy, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

DATA_DIR = Path('.')  # Change if data are on Google Drive.
OUTPUT_DIR = DATA_DIR / 'artifacts'
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
SEED = 42
BATCH_SIZE = 64
EPOCHS = 18
WARMUP_EPOCHS = 3
NUM_WORKERS = 0  # Portable to Windows, macOS and Colab.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
def seed_everything():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()
print('Device:', DEVICE, '| Python:', sys.version, '| PyTorch:', torch.__version__)
if DEVICE.type == 'cpu': print('Use a Colab GPU for the ResNet training cells.')


## 1. Inspect and validate the dataset
Read IDs as strings to retain leading zeros. Match IDs to filenames with their extensions. Composite transparency onto white consistently for both splits. Do not use test images or Kaggle results to select training settings.


In [ ]:
def image_map(folder):
    paths = sorted(p for p in folder.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'})
    mapping = {p.stem: p for p in paths}
    assert len(mapping) == len(paths), 'Duplicate image stems'
    return mapping
labels = pd.read_csv(DATA_DIR / 'train_labels.csv', dtype={'Id': str})
assert list(labels.columns) == ['Id', 'Label']
assert labels.notna().all().all() and labels.Id.is_unique
train_paths = image_map(DATA_DIR / 'train')
test_paths = image_map(DATA_DIR / 'test')
assert set(labels.Id) == set(train_paths), 'Training files do not match labels'
assert len(test_paths) > 0 and not (set(train_paths) & set(test_paths))
classes = sorted(str(x) for x in labels.Label.unique())
class_to_idx = {name:i for i,name in enumerate(classes)}
print('Train:', len(labels), 'Test:', len(test_paths), 'Providers:', classes)
display(labels.head())
display(labels.Label.value_counts().rename('images').to_frame())
labels.Label.value_counts().plot.bar(title='Training images by provider'); plt.show()

def open_rgb(path):
    with Image.open(path) as im:
        rgba = im.convert('RGBA')
        return Image.alpha_composite(Image.new('RGBA', rgba.size, 'white'), rgba).convert('RGB')
fig, axes = plt.subplots(2, 7, figsize=(15, 5))
for ax, (_, row) in zip(axes.flat, labels.sample(14, random_state=SEED).iterrows()):
    ax.imshow(open_rgb(train_paths[row.Id])); ax.set_title(row.Label); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
train_idx, val_idx = train_test_split(np.arange(len(labels)), test_size=0.2,
    random_state=SEED, stratify=labels.Label)
assert not set(train_idx) & set(val_idx)
pd.DataFrame({'Id': labels.Id, 'split': np.where(np.isin(np.arange(len(labels)), val_idx), 'validation', 'train')}).to_csv(OUTPUT_DIR / 'split.csv', index=False)
print(len(train_idx), 'training /', len(val_idx), 'validation')


## 2. Explainable CPU baseline
Extra Trees uses small RGB thumbnails, transparency and color histograms. It checks the entire data-to-submission pipeline without pretrained weights. Its split uses the same seed and indices as the neural network. Running it here creates `submission_baseline.csv`; you can submit that file while the neural network trains.


In [ ]:
# baseline.py must be alongside this notebook. It also runs independently.
import subprocess
subprocess.run([sys.executable, str(Path('baseline.py').resolve()), str(DATA_DIR.resolve())], check=True)
baseline_metrics = json.loads((OUTPUT_DIR / 'baseline_metrics.json').read_text())
print('Baseline validation accuracy:', baseline_metrics['validation_accuracy'])


**Development run:** the CPU baseline achieved 82.14% validation accuracy on the provided data (seed 42, 80/20 stratified split). Full metrics are in `artifacts/baseline_metrics.json`. This notebook has not yet been executed end to end; ResNet performance must be measured by running the training cells.


## 3. Pretrained ResNet18
ImageNet pretraining provides useful edges and texture features. Replace the classifier with the provider classes. Warm up the classifier for three epochs, then fine-tune the whole network at a lower learning rate. Use mild geometric augmentation; keep colors because they may identify a provider. Avoid flips that reverse emoji text. Resize the full image without cropping.

The first model construction downloads ImageNet weights and requires internet. Confirm that external pretrained weights are permitted by your competition rules before running this section.


In [ ]:
SIZE = 128
normalize = transforms.Normalize([.485,.456,.406], [.229,.224,.225])
train_transform = transforms.Compose([
    transforms.Resize((SIZE,SIZE)),
    transforms.RandomAffine(degrees=5, translate=(.04,.04), scale=(.95,1.05), fill=255),
    transforms.ToTensor(), normalize])
eval_transform = transforms.Compose([transforms.Resize((SIZE,SIZE)), transforms.ToTensor(), normalize])
class EmojiDataset(Dataset):
    def __init__(self, ids, mapping, transform, targets=None):
        self.ids=list(ids); self.mapping=mapping; self.transform=transform; self.targets=targets
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        x=self.transform(open_rgb(self.mapping[self.ids[i]]))
        return (x, int(self.targets[i])) if self.targets is not None else x

y = labels.Label.map(class_to_idx).to_numpy()
def loader(indices, training):
    ds = EmojiDataset(labels.Id.iloc[indices], train_paths,
                      train_transform if training else eval_transform, y[indices])
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=training,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type=='cuda')
train_loader = loader(train_idx, True)
val_loader = loader(val_idx, False)
def make_model():
    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc=nn.Linear(model.fc.in_features, len(classes))
    return model.to(DEVICE)

def prepare_stage(model, epoch):
    warmup = epoch < WARMUP_EPOCHS
    for p in model.parameters(): p.requires_grad = not warmup
    for p in model.fc.parameters(): p.requires_grad = True
    return torch.optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()),
                             lr=1e-3 if warmup else 1e-4, weight_decay=1e-4)

criterion=nn.CrossEntropyLoss()
def train_epoch(model, batches, optimizer, warmup):
    model.train()
    if warmup:
        # Frozen backbone includes BatchNorm running statistics.
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d): module.eval()
    total_loss=0.; correct=0; count=0
    for images, targets in batches:
        images, targets=images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits=model(images); loss=criterion(logits,targets)
        loss.backward(); optimizer.step()
        total_loss+=loss.item()*len(targets)
        correct+=(logits.argmax(1)==targets).sum().item(); count+=len(targets)
    return total_loss/count, correct/count

@torch.inference_mode()
def evaluate(model,batches):
    model.eval(); predictions=[]; truth=[]; loss_sum=0
    for images, targets in batches:
        logits=model(images.to(DEVICE))
        loss_sum+=criterion(logits,targets.to(DEVICE)).item()*len(targets)
        predictions.extend(logits.argmax(1).cpu().tolist()); truth.extend(targets.tolist())
    return loss_sum/len(truth), accuracy_score(truth,predictions), np.array(predictions)


In [ ]:
seed_everything()
model=make_model()
best_accuracy=-1.; best_loss=float('inf'); best_epoch=0; history=[]
for epoch in range(EPOCHS):
    if epoch in (0,WARMUP_EPOCHS): optimizer=prepare_stage(model,epoch)
    train_loss,train_accuracy=train_epoch(model,train_loader,optimizer,epoch<WARMUP_EPOCHS)
    val_loss,val_accuracy,_=evaluate(model,val_loader)
    history.append(dict(epoch=epoch+1,train_loss=train_loss,train_accuracy=train_accuracy,
                        val_loss=val_loss,val_accuracy=val_accuracy))
    if val_accuracy>best_accuracy or (val_accuracy==best_accuracy and val_loss<best_loss):
        best_accuracy,best_loss,best_epoch=val_accuracy,val_loss,epoch+1
        torch.save({'state_dict':model.state_dict(),'classes':classes,'epoch':best_epoch,
                    'validation_accuracy':best_accuracy,'size':SIZE}, OUTPUT_DIR/'best_resnet18.pt')
    pd.DataFrame(history).to_csv(OUTPUT_DIR/'training_history.csv', index=False)
    print(f'{epoch+1:02d}/{EPOCHS}: train={train_accuracy:.4f}, validation={val_accuracy:.4f}',flush=True)
print('Best epoch:',best_epoch,'| validation accuracy:',best_accuracy)


## 4. Validation and error analysis
Evaluate the selected checkpoint on the held-out images. This split guided epoch selection, so its accuracy is a model-selection estimate. Inspect per-provider recall and mistakes before deciding what to try next. Similar emoji content can occur across splits; a semantic-group split would be a tougher test of generalization to unseen emoji designs.


In [ ]:
checkpoint=torch.load(OUTPUT_DIR/'best_resnet18.pt',map_location=DEVICE,weights_only=True)
model.load_state_dict(checkpoint['state_dict'])
_,resnet_accuracy,val_prediction=evaluate(model,val_loader)
print(classification_report(y[val_idx],val_prediction,labels=np.arange(len(classes)),target_names=classes,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y[val_idx],val_prediction,display_labels=classes,xticks_rotation=45)
plt.tight_layout(); plt.show()
pd.DataFrame(history).set_index('epoch')[['train_accuracy','val_accuracy']].plot(); plt.show()
errors=val_idx[val_prediction!=y[val_idx]]
fig,axes=plt.subplots(2,5,figsize=(14,6))
for ax in axes.flat: ax.axis('off')
for ax,idx in zip(axes.flat,errors[:10]):
    pred=val_prediction[np.flatnonzero(val_idx==idx)[0]]
    ax.imshow(open_rgb(train_paths[labels.Id.iloc[idx]]))
    ax.set_title(f'True: {labels.Label.iloc[idx]}\nPred: {classes[pred]}')
plt.tight_layout(); plt.show()
metrics={'resnet_validation_accuracy':resnet_accuracy,'baseline_validation_accuracy':baseline_metrics['validation_accuracy'],
         'best_epoch':best_epoch,'seed':SEED,'classes':classes}
(OUTPUT_DIR/'model_comparison.json').write_text(json.dumps(metrics,indent=2))
display(pd.DataFrame({'model':['Extra Trees','ResNet18'], 'validation_accuracy':[baseline_metrics['validation_accuracy'],resnet_accuracy]}))


## 5. Refit and predict
Refit ResNet on all labels for the number of epochs selected above. Validation images now contribute to the final model: do not report accuracy measured on this final training set as validation performance. This step costs another training run. The baseline already refits on all labels.


In [ ]:
seed_everything()
final_model=make_model()
all_loader=loader(np.arange(len(labels)),True)
for epoch in range(best_epoch):
    if epoch in (0,WARMUP_EPOCHS): optimizer=prepare_stage(final_model,epoch)
    loss,accuracy=train_epoch(final_model,all_loader,optimizer,epoch<WARMUP_EPOCHS)
    print(f'Full-data epoch {epoch+1}/{best_epoch}: loss={loss:.4f}',flush=True)
torch.save({'state_dict':final_model.state_dict(),'classes':classes,'size':SIZE,'epochs':best_epoch}, OUTPUT_DIR/'final_resnet18.pt')


In [ ]:
test_ids=sorted(test_paths)
test_loader=DataLoader(EmojiDataset(test_ids,test_paths,eval_transform),batch_size=BATCH_SIZE,
                       shuffle=False,num_workers=NUM_WORKERS)
final_model.eval(); predicted=[]
with torch.inference_mode():
    for images in test_loader:
        predicted.extend(final_model(images.to(DEVICE)).argmax(1).cpu().tolist())
result=pd.DataFrame({'Id':test_ids,'Label':[classes[i] for i in predicted]})
sample_path=DATA_DIR/'sample_submission.csv'
if sample_path.exists():
    sample=pd.read_csv(sample_path,dtype={'Id':str})
    assert list(sample.columns)==['Id','Label'], 'Adapt output columns to the official template'
    assert set(sample.Id)==set(result.Id)
    result=sample[['Id']].merge(result,on='Id',validate='one_to_one')
assert len(result)==len(test_paths) and result.Id.is_unique
assert result.notna().all().all() and set(result.Label)<=set(classes)
result.to_csv(DATA_DIR/'submission_resnet18.csv',index=False)
# Pick by validation only, before inspecting any Kaggle score.
chosen = 'submission_resnet18.csv' if resnet_accuracy > baseline_metrics['validation_accuracy'] else 'submission_baseline.csv'
import shutil
shutil.copyfile(DATA_DIR/chosen,DATA_DIR/'submission.csv')
print('Selected:',chosen,'→ submission.csv')
display(pd.read_csv(DATA_DIR/'submission.csv',dtype={'Id':str}).head())


## 6. Download and submit
Upload `submission.csv` in the Kaggle competition's submission form. Check column names against the official sample submission; the invitation page could not be read during development. The file contains textual provider labels and one row per test image. A leading zero displayed differently by Excel does not mean the CSV is wrong: inspect it as text.


In [ ]:
try:
    from google.colab import files
    files.download(str(DATA_DIR/'submission.csv'))
except ImportError:
    print('Local file:',(DATA_DIR/'submission.csv').resolve())


## AI usage and your own report
This code and notebook structure were generated with OpenAI Codex assistance. See `codex-AI_USAGE.md` for the request and scope. The team must review the code, run the notebook in Colab, understand the models, and write the report independently. Include actual validation metrics, Kaggle scores, hardware, package versions, experiments, limitations, and each teammate's contribution. No Kaggle score is asserted by this notebook.
